# 06 · Boosting, Optuna e SHAP

**Programação para Aprendizagem de Máquina** · Especialização em Ciência de Dados

Na aula 05 montamos um pipeline completo com regressão logística. Agora trocamos o motor: o **gradient boosting**, a família de modelos que mais vence competições e mais aparece em produção quando os dados são tabelas. Junto com ele vêm duas ferramentas que fazem parte do mesmo ofício: o **Optuna**, para procurar hiperparâmetros, e o **SHAP**, para explicar o que o modelo aprendeu.

## O que você será capaz de fazer ao final

1. Explicar como o boosting constrói um modelo forte somando árvores pequenas.
2. Treinar modelos com `HistGradientBoostingClassifier` e LightGBM, com parada antecipada.
3. Reconhecer sobreajuste nas curvas de treino e validação, e saber quais hiperparâmetros o controlam.
4. Escrever uma busca de hiperparâmetros com o Optuna e avaliar se o ganho compensa.
5. Comparar três formas de medir a importância das colunas e interpretar gráficos do SHAP.
6. Dizer quando o boosting é a escolha certa e quando não é.

## Como usar este notebook

| Símbolo | Significado |
|---|---|
| 📌 | Lembrete de algo que você já viu e vai precisar agora |
| 🔮 | **Preveja antes de rodar**: responda mentalmente (ou por escrito) antes de executar a próxima célula |
| ✏️ | Exercício. A dificuldade vai de ★ (fixação) a ★★★ (desafio) |
| ✅ | Célula de verificação: confere se a sua resposta faz sentido. Não altere |
| ⚠️ | Armadilha comum |
| ⏳ | Célula que demora mais de 20 segundos |

Nos exercícios, substitua `...` pelo seu código. Se a célula ✅ terminar sem erro, você pode seguir. Se você travar em um exercício, pule: nenhuma célula posterior depende dele.

In [ ]:
import importlib.util
import subprocess
import sys

faltando = [pacote for pacote in ("lightgbm", "optuna", "shap") if importlib.util.find_spec(pacote) is None]
if faltando:                                   # no Colab, optuna e shap precisam ser instalados
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *faltando], check=True)

import inspect
import time
import warnings
from pathlib import Path
from urllib.request import urlretrieve

warnings.filterwarnings("ignore", message="IProgress not found")

import lightgbm as lgb
import matplotlib.pyplot as plt
import numpy as np
import optuna
import pandas as pd
import shap
import sklearn
from sklearn.metrics import roc_auc_score

print("scikit-learn", sklearn.__version__, "· lightgbm", lgb.__version__, "· optuna", optuna.__version__, "· shap", shap.__version__)
assert tuple(int(p) for p in sklearn.__version__.split(".")[:2]) >= (1, 4), "Esta aula precisa do scikit-learn 1.4 ou mais novo."

SEMENTE = 42
AZUL, LARANJA, VERDE, CINZA = "#2a78d6", "#eb6834", "#1baf7a", "#6b6b6b"
optuna.logging.set_verbosity(optuna.logging.WARNING)          # o Optuna imprime uma linha por tentativa; silenciamos
warnings.filterwarnings("ignore", message="X does not have valid feature names")
pd.set_option("display.max_colwidth", 110)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.titlelocation": "left", "axes.titleweight": "bold", "axes.titlesize": 11})

## 1. Ponto de partida

Usamos os mesmos dados e o mesmo alvo da aula 05: prever se o cliente **não recomenda** o produto. A preparação é a mesma, com três colunas novas tiradas do texto por contagem simples.

In [ ]:
URL = (
    "https://raw.githubusercontent.com/americanas-tech/b2w-reviews01/"
    "4639429ec698d7821fc99a0bc665fa213d9fcd5a/B2W-Reviews01.csv"
)
ARQUIVO = Path("B2W-Reviews01.csv")
if not ARQUIVO.exists():
    urlretrieve(URL, ARQUIVO)

bruto = pd.read_csv(ARQUIVO, dtype={"product_id": str}, low_memory=False)
df = bruto.dropna(subset=["recommend_to_a_friend"]).copy()
df["nao_recomenda"] = (df["recommend_to_a_friend"] == "No").astype(int)

df["submission_date"] = pd.to_datetime(df["submission_date"])
idade = 2018 - df["reviewer_birth_year"]
df["idade"] = idade.where(idade.between(14, 100))
df["hora"] = df["submission_date"].dt.hour
df["dia_semana"] = df["submission_date"].dt.dayofweek
df["texto"] = df["review_title"].fillna("") + ". " + df["review_text"].fillna("")

df["palavras"] = df["texto"].str.split().str.len()                               # tamanho do texto
df["exclamacoes"] = df["texto"].str.count("!")                                   # quantos pontos de exclamação
df["maiusculas"] = df["texto"].str.count(r"[A-ZÀ-Ý]") / df["texto"].str.len()    # proporção de letras maiúsculas

NUMERICAS = ["idade", "hora", "dia_semana", "palavras", "exclamacoes", "maiusculas"]
CATEGORICAS = ["site_category_lv1", "site_category_lv2", "product_brand", "reviewer_state", "reviewer_gender"]

X = df.drop(columns=["nao_recomenda", "recommend_to_a_friend"])
y = df["nao_recomenda"]

### Três conjuntos

Na aula 05 separamos o **teste** (o cofre) e, dentro do treino, usamos validação cruzada. Aqui vamos dividir o treino mais uma vez, em **ajuste** e **validação**, por dois motivos:

- o boosting usa a validação **durante** o treino, para decidir quando parar (seção 5);
- treinaremos dezenas de modelos, e uma validação fixa custa um quinto de uma validação cruzada de 5 partes.

O preço é uma estimativa um pouco mais ruidosa. O teste continua trancado até a seção 9.

In [ ]:
from sklearn.model_selection import train_test_split

X_treino, X_teste, y_treino, y_teste = train_test_split(X, y, test_size=0.2, stratify=y, random_state=SEMENTE)      # igual à aula 05
X_ajuste, X_valid, y_ajuste, y_valid = train_test_split(X_treino, y_treino, test_size=0.25, stratify=y_treino, random_state=SEMENTE)

for nome, parte in [("ajuste", y_ajuste), ("validação", y_valid), ("teste", y_teste)]:
    print(f"{nome:<10}{len(parte):>7} linhas · {parte.mean():.1%} não recomendam")

### As referências da aula 05

Dois modelos lineares servem de régua: um só com as colunas tabulares e outro só com o texto (TF-IDF).

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


def auc(modelo, X_, y_):
    """ROC AUC de um modelo já treinado."""
    return roc_auc_score(y_, modelo.predict_proba(X_)[:, 1])


logistica_tabular = make_pipeline(
    ColumnTransformer([
        ("num", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), NUMERICAS),
        ("cat", OneHotEncoder(handle_unknown="ignore", min_frequency=20), CATEGORICAS),
    ]),
    LogisticRegression(max_iter=1000),
).fit(X_ajuste, y_ajuste)

logistica_texto = make_pipeline(
    ColumnTransformer([("txt", TfidfVectorizer(min_df=5), "texto")]),
    LogisticRegression(max_iter=1000),
).fit(X_ajuste, y_ajuste)

resultados = {
    "logística · tabular": auc(logistica_tabular, X_valid, y_valid),
    "logística · texto (TF-IDF)": auc(logistica_texto, X_valid, y_valid),
}
for nome, valor in resultados.items():
    print(f"{nome:<30}{valor:.4f}")

Estes são os números a bater: cerca de 0,73 com as colunas tabulares e 0,96 com o texto.

## 2. A ideia do boosting

Uma árvore de decisão pequena erra muito. O **boosting** combina centenas delas, com uma regra simples:

> cada árvore nova é treinada para corrigir os **erros** que o conjunto das anteriores ainda comete.

Para ver isso funcionando, vamos a um problema de brinquedo com uma única variável.

In [ ]:
from sklearn.tree import DecisionTreeRegressor

rng = np.random.default_rng(0)
x = np.sort(rng.uniform(0, 10, 200))
y_brinquedo = np.sin(x) + 0.1 * x + rng.normal(0, 0.3, 200)       # uma curva com ruído
X_brinquedo = x.reshape(-1, 1)

TAXA = 0.3                                                          # taxa de aprendizado

previsao_0 = np.full(len(x), y_brinquedo.mean())                    # rodada 0: prever a média para todos

residuo_1 = y_brinquedo - previsao_0                                # o que falta acertar
arvore_1 = DecisionTreeRegressor(max_depth=2, random_state=0).fit(X_brinquedo, residuo_1)     # árvore treinada NO RESÍDUO
previsao_1 = previsao_0 + TAXA * arvore_1.predict(X_brinquedo)      # dá um passo pequeno na direção da correção

residuo_2 = y_brinquedo - previsao_1
arvore_2 = DecisionTreeRegressor(max_depth=2, random_state=0).fit(X_brinquedo, residuo_2)
previsao_2 = previsao_1 + TAXA * arvore_2.predict(X_brinquedo)

fig, eixos = plt.subplots(1, 3, figsize=(12, 3.2), sharey=True)
for ax, previsao, rodada in zip(eixos, [previsao_0, previsao_1, previsao_2], [0, 1, 2]):
    ax.scatter(x, y_brinquedo, s=8, color=CINZA, alpha=0.5)
    ax.plot(x, previsao, color=LARANJA, lw=2)
    ax.set_title(f"Depois de {rodada} árvore(s): erro quadrático médio {np.mean((y_brinquedo - previsao) ** 2):.3f}")
    ax.set_xlabel("x")
plt.tight_layout()

Três elementos definem o algoritmo:

| Elemento | Papel |
|---|---|
| **Resíduo** | o alvo de cada árvore nova: valor verdadeiro menos previsão atual |
| **Árvores rasas** | cada uma corrige um pouco; sozinha, seria um modelo fraco |
| **Taxa de aprendizado** | fração da correção que é aplicada a cada rodada; passos pequenos tornam o processo mais estável |

A previsão final é a soma: `média + taxa × (árvore 1 + árvore 2 + ... + árvore N)`.

### ✏️ Exercício 1 (★★): boosting em dez linhas

Generalize o que foi feito acima para qualquer número de árvores.

- `treinar_boosting` devolve a previsão inicial (a média de `y`) e a lista de árvores treinadas, cada uma no resíduo deixado pelas anteriores. Use `DecisionTreeRegressor(max_depth=profundidade, random_state=0)`.
- `prever_boosting` devolve as previsões para `X`.

In [ ]:
# ✏️ Exercício 1
def treinar_boosting(X, y, n_arvores, taxa, profundidade=2):
    # SEU CÓDIGO AQUI
    ...
    return inicial, arvores


def prever_boosting(inicial, arvores, taxa, X):
    # SEU CÓDIGO AQUI
    ...

In [ ]:
# ✅ Verificação do exercício 1 (não altere)
from sklearn.ensemble import GradientBoostingRegressor

_inicial, _arvores = treinar_boosting(X_brinquedo, y_brinquedo, n_arvores=100, taxa=0.3)
_minha = prever_boosting(_inicial, _arvores, 0.3, X_brinquedo)
_oficial = GradientBoostingRegressor(n_estimators=100, learning_rate=0.3, max_depth=2, random_state=0).fit(X_brinquedo, y_brinquedo).predict(X_brinquedo)
assert len(_arvores) == 100 and np.isclose(_inicial, y_brinquedo.mean())
assert np.allclose(_minha, _oficial), "as previsões diferem das do scikit-learn: cada árvore foi treinada no resíduo atual?"
print(f"Exercício 1 correto: suas previsões coincidem com as do GradientBoostingRegressor (diferença máxima {np.abs(_minha - _oficial).max():.1e}).")

### Quantas árvores? O risco de decorar o ruído

Se cada árvore corrige os erros da anterior, por que não usar milhares? Porque, a partir de certo ponto, o que resta para corrigir é **ruído**.

🔮 **Preveja:** com 2000 árvores, o que acontece com o erro no conjunto de **ajuste**? E em pontos novos, tirados da mesma curva?

In [ ]:
x_novo = np.sort(rng.uniform(0, 10, 200))                           # pontos novos da mesma curva
y_novo = np.sin(x_novo) + 0.1 * x_novo + rng.normal(0, 0.3, 200)

curvas = {}
for taxa in [1.0, 0.1]:
    modelo = GradientBoostingRegressor(n_estimators=2000, learning_rate=taxa, max_depth=2, random_state=0).fit(X_brinquedo, y_brinquedo)
    curvas[taxa] = (
        [np.mean((y_brinquedo - p) ** 2) for p in modelo.staged_predict(X_brinquedo)],              # erro após cada árvore
        [np.mean((y_novo - p) ** 2) for p in modelo.staged_predict(x_novo.reshape(-1, 1))],
    )

fig, eixos = plt.subplots(1, 2, figsize=(11, 3.4), sharey=True)
for ax, taxa in zip(eixos, curvas):
    erro_ajuste, erro_novo = curvas[taxa]
    melhor = int(np.argmin(erro_novo)) + 1
    ax.plot(range(1, 2001), erro_ajuste, color=CINZA, label="nos dados de ajuste")
    ax.plot(range(1, 2001), erro_novo, color=LARANJA, label="em dados novos")
    ax.axvline(melhor, color=LARANJA, ls=":", lw=1)
    ax.axhline(0.09, color=CINZA, ls="--", lw=0.8)
    ax.set_xscale("log"); ax.set_xlabel("número de árvores (escala logarítmica)")
    ax.set_title(f"Taxa {taxa}: melhor com {melhor} árvores (erro {min(erro_novo):.3f})")
eixos[0].set_ylabel("erro quadrático médio"); eixos[0].legend(frameon=False)
eixos[1].text(1.1, 0.065, "variância do ruído (0,09)", fontsize=8, color=CINZA)
plt.tight_layout()
print({taxa: f"erro em dados novos com 2000 árvores: {curvas[taxa][1][-1]:.3f}" for taxa in curvas})

**O que o gráfico mostra.**

- O erro no ajuste **cai sempre**: com árvores suficientes, o modelo decora cada ponto, ruído incluído.
- O erro em dados novos cai, chega a um mínimo e **volta a subir**. É o sobreajuste. O melhor que se pode esperar é ficar perto da linha tracejada, a variância do ruído.
- Com taxa menor, o mínimo chega mais tarde (são necessárias mais árvores), a curva é mais suave e o mínimo é um pouco mais baixo.

Disso sai a receita usada na prática: **taxa de aprendizado pequena, muitas árvores e parada automática** quando o erro de validação deixa de melhorar. É a **parada antecipada** (*early stopping*), que veremos na seção 5.

## 3. Da regressão à classificação

Em classificação, o modelo soma árvores em **log-chances** (o mesmo espaço em que a regressão logística trabalha) e converte o total em probabilidade com a função sigmoide. O "resíduo" de cada linha passa a ser `y − p`: a diferença entre a classe verdadeira (0 ou 1) e a probabilidade prevista.

O nome **gradient** boosting vem daí. Esse resíduo é o gradiente (com sinal trocado) da função de perda em relação à previsão. Trocar a função de perda troca o resíduo, e o mesmo algoritmo serve para regressão, classificação, ordenação e outras tarefas.

### Boosting e floresta aleatória

| | Floresta aleatória (*bagging*) | Gradient boosting |
|---|---|---|
| Como as árvores são treinadas | independentes, em paralelo, cada uma em uma amostra dos dados | em sequência, cada uma corrigindo as anteriores |
| Árvores | profundas | rasas |
| O que a combinação reduz | variância | viés |
| Mais árvores | nunca prejudica | pode sobreajustar |
| Ajuste de hiperparâmetros | pouco sensível | mais sensível, e costuma chegar mais longe |

## 4. As implementações modernas

O `GradientBoostingClassifier` clássico do scikit-learn avalia todos os pontos de corte possíveis e fica lento com muitas linhas. As implementações atuais agrupam cada coluna numérica em até 255 **faixas** (um histograma) e só avaliam cortes entre faixas. O ganho de velocidade é de dezenas de vezes, quase sem perda de qualidade.

| Biblioteca | Observações |
|---|---|
| `HistGradientBoostingClassifier` (scikit-learn) | já vem instalado; boa primeira opção |
| **LightGBM** | muito rápido; árvores que crescem folha a folha |
| **XGBoost** | o mais antigo dos três; ecossistema amplo; suporte a GPU |
| **CatBoost** | tratamento cuidadoso de colunas categóricas |

As diferenças de qualidade entre elas costumam ser menores que as produzidas por uma boa escolha de colunas. Nesta aula usamos as duas primeiras.

### Preparação para árvores

Árvores pedem menos preparo que modelos lineares:

| Etapa | Modelo linear | Boosting moderno |
|---|---|---|
| Padronizar colunas numéricas | necessário | **desnecessário**: só a ordem dos valores importa |
| Preencher ausentes | necessário | **desnecessário**: os ausentes seguem pelo ramo que mais reduz o erro |
| Categóricas | *one-hot* | um código inteiro por categoria é suficiente |

In [ ]:
from sklearn.preprocessing import OrdinalEncoder


def codificador():
    """Código inteiro por categoria; as raras são agrupadas e as desconhecidas viram ausentes."""
    return OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=np.nan, max_categories=60)


def nomes_das_colunas(pre):
    """Nomes legíveis para as colunas que saem do ColumnTransformer."""
    return ["palavra: " + nome[5:] if nome.startswith("txt__") else nome[5:] for nome in pre.get_feature_names_out()]


pre_tabular = ColumnTransformer([("num", "passthrough", NUMERICAS), ("cat", codificador(), CATEGORICAS)])

A_tab = pre_tabular.fit_transform(X_ajuste)            # aprende os códigos SÓ no ajuste
V_tab = pre_tabular.transform(X_valid)
nomes_tab = nomes_das_colunas(pre_tabular)

print(A_tab.shape, "· ausentes na matriz:", int(np.isnan(A_tab).sum()))
pd.DataFrame(A_tab[:3], columns=nomes_tab).round(2)

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier

inicio = time.perf_counter()
hgb = HistGradientBoostingClassifier(random_state=SEMENTE).fit(A_tab, y_ajuste)          # tudo no padrão

resultados["boosting (HGB) · tabular"] = auc(hgb, V_tab, y_valid)
print(f"HistGradientBoosting: AUC {resultados['boosting (HGB) · tabular']:.4f} em {time.perf_counter() - inicio:.1f} s, com {hgb.n_iter_} árvores")
print(f"Regressão logística:  AUC {resultados['logística · tabular']:.4f}")

Sem padronizar, sem preencher ausentes e sem ajustar nada, o boosting supera a regressão logística por quase quatro centésimos de AUC nas mesmas colunas. A vantagem vem do que as árvores capturam sozinhas e o modelo linear não: efeitos **não lineares** (a relação com o tamanho do texto não é uma reta) e **interações** entre colunas (uma categoria que só é problemática em certo estado, por exemplo).

## 5. LightGBM: parada antecipada e sobreajuste

O LightGBM segue a interface do scikit-learn (`fit`, `predict_proba`). Os hiperparâmetros que mais importam:

| Hiperparâmetro | O que controla | Efeito de aumentar |
|---|---|---|
| `n_estimators` | número de árvores | mais capacidade; deixe alto e use parada antecipada |
| `learning_rate` | tamanho do passo | aprende mais rápido, com risco de passar do ponto |
| `num_leaves` | folhas por árvore | árvores mais complexas |
| `min_child_samples` | mínimo de linhas em uma folha | árvores **mais simples** (regulariza) |
| `colsample_bytree`, `subsample` | fração de colunas e de linhas sorteada por árvore | abaixo de 1 regulariza e acelera |
| `reg_lambda` | penalidade sobre os valores das folhas | regulariza |

### Parada antecipada

Entregamos ao `fit` o conjunto de validação. A cada árvore, o LightGBM mede nele a perda logarítmica; se não houver melhora por 50 árvores seguidas, o treino para e o modelo guarda a melhor iteração.

In [ ]:
def avaliar_em(*pares):
    """Monta os argumentos de avaliação do `fit`. (O nome do argumento mudou em versões recentes do LightGBM.)"""
    if "eval_X" in inspect.signature(lgb.LGBMClassifier.fit).parameters:
        return {"eval_X": tuple(X_ for X_, _ in pares), "eval_y": tuple(y_ for _, y_ in pares)}
    return {"eval_set": list(pares)}


def treinar_lgbm(A, V, taxa=0.1, paciencia=50, **parametros):
    """Treina um LightGBM com parada antecipada, acompanhando a perda no ajuste e na validação."""
    modelo = lgb.LGBMClassifier(n_estimators=3000, learning_rate=taxa, random_state=SEMENTE, verbose=-1, **parametros)
    modelo.fit(
        A, y_ajuste,
        **avaliar_em((A, y_ajuste), (V, y_valid)), eval_names=["ajuste", "validação"],
        callbacks=[lgb.early_stopping(paciencia, verbose=False)],        # para quando a VALIDAÇÃO deixa de melhorar
    )
    return modelo


inicio = time.perf_counter()
lgbm_tab = treinar_lgbm(A_tab, V_tab)
resultados["boosting (LightGBM) · tabular"] = auc(lgbm_tab, V_tab, y_valid)

print(f"Limite de 3000 árvores; melhor iteração: {lgbm_tab.best_iteration_} · AUC de validação {resultados['boosting (LightGBM) · tabular']:.4f} · {time.perf_counter() - inicio:.1f} s")

In [ ]:
def grafico_das_curvas(modelo, ax, titulo):
    historico = modelo.evals_result_
    ax.plot(historico["ajuste"]["binary_logloss"], color=CINZA, label="ajuste")
    ax.plot(historico["validação"]["binary_logloss"], color=LARANJA, label="validação")
    if getattr(modelo, "best_iteration_", None):
        ax.axvline(modelo.best_iteration_, color=LARANJA, ls=":", lw=1)
    ax.set_xlabel("número de árvores"); ax.set_ylabel("perda logarítmica"); ax.set_title(titulo); ax.legend(frameon=False)


fig, ax = plt.subplots(figsize=(6.5, 3.4))
grafico_das_curvas(lgbm_tab, ax, "A perda de validação se estabiliza enquanto a de ajuste continua caindo")

A distância entre as duas curvas é o sobreajuste se instalando. A linha pontilhada marca a melhor iteração; sem a parada antecipada, o treino seguiria até a árvore 3000 gastando tempo para piorar.

> ⚠️ **A validação usada para parar o treino já não é uma estimativa imparcial.** O número de árvores foi escolhido olhando para ela. O viés costuma ser pequeno, mas é mais um motivo para manter o teste intocado.

### Taxa de aprendizado e número de árvores

In [ ]:
linhas = []
for taxa in [0.3, 0.1, 0.03]:
    inicio = time.perf_counter()
    modelo = treinar_lgbm(A_tab, V_tab, taxa=taxa)
    linhas.append({"taxa": taxa, "árvores": modelo.best_iteration_, "AUC validação": auc(modelo, V_tab, y_valid), "segundos": time.perf_counter() - inicio})
pd.DataFrame(linhas).round({"AUC validação": 4, "segundos": 1})

Reduzir a taxa dez vezes (de 0,3 para 0,03) multiplicou o número de árvores por quase quinze e o tempo por perto de dez, para um ganho na terceira casa decimal. O compromisso usual: taxa em torno de **0,1 durante a exploração**; uma taxa menor apenas no treino final, se o ganho compensar.

### Como é um modelo sobreajustado

Vamos soltar os freios: árvores com até 255 folhas, folhas com uma única linha, 300 árvores e nenhuma parada antecipada.

In [ ]:
solto = lgb.LGBMClassifier(n_estimators=300, learning_rate=0.1, num_leaves=255, min_child_samples=1, random_state=SEMENTE, verbose=-1)
solto.fit(A_tab, y_ajuste, **avaliar_em((A_tab, y_ajuste), (V_tab, y_valid)), eval_names=["ajuste", "validação"])

print(f"Modelo solto:          AUC no ajuste {auc(solto, A_tab, y_ajuste):.3f} · na validação {auc(solto, V_tab, y_valid):.3f}")
print(f"Com parada antecipada: AUC no ajuste {auc(lgbm_tab, A_tab, y_ajuste):.3f} · na validação {auc(lgbm_tab, V_tab, y_valid):.3f}")

fig, ax = plt.subplots(figsize=(6.5, 3.4))
grafico_das_curvas(solto, ax, "Sem freios: a perda de validação volta a subir cedo")

O modelo solto quase acerta tudo no ajuste (AUC de 0,99) e é **pior** na validação que o modelo contido. A assinatura do sobreajuste é essa combinação: desempenho de ajuste excelente, distância grande para a validação, e curva de validação que piora com mais árvores (aqui, a partir de umas 40).

### ✏️ Exercício 2 (★★): um modelo mais contido

Use `treinar_lgbm` para treinar `modelo_ex2` na matriz tabular com `taxa=0.05`, `num_leaves=15` e `min_child_samples=100`. Guarde em `auc_ajuste_ex2` e `auc_valid_ex2` as AUC nos dois conjuntos.

In [ ]:
# ✏️ Exercício 2
# SEU CÓDIGO AQUI
modelo_ex2 = ...
auc_ajuste_ex2 = ...
auc_valid_ex2 = ...

In [ ]:
# ✅ Verificação do exercício 2 (não altere)
_p = modelo_ex2.get_params()
assert (_p["num_leaves"], _p["min_child_samples"], _p["learning_rate"]) == (15, 100, 0.05), "confira os três hiperparâmetros pedidos"
assert modelo_ex2.best_iteration_ and modelo_ex2.best_iteration_ < 3000, "o modelo deve ter parado antes do limite de árvores"
assert 0.75 < auc_valid_ex2 < auc_ajuste_ex2 < 0.90
print(f"Exercício 2 correto: {modelo_ex2.best_iteration_} árvores · AUC no ajuste {auc_ajuste_ex2:.3f} · na validação {auc_valid_ex2:.3f} · distância {auc_ajuste_ex2 - auc_valid_ex2:.3f}")

### ✏️ Exercício 3 (★★): complexidade das árvores

Treine três modelos `lgb.LGBMClassifier` na matriz tabular, com `n_estimators=200`, `learning_rate=0.1`, `random_state=SEMENTE`, `verbose=-1`, **sem** parada antecipada, variando `num_leaves` em 4, 31 e 255. Monte `tabela_ex3`, um DataFrame com as colunas `num_leaves`, `auc_ajuste` e `auc_valid`, uma linha por modelo, na ordem dada.

In [ ]:
# ✏️ Exercício 3
# SEU CÓDIGO AQUI
tabela_ex3 = ...

In [ ]:
# ✅ Verificação do exercício 3 (não altere)
assert list(tabela_ex3.columns) == ["num_leaves", "auc_ajuste", "auc_valid"] and tabela_ex3["num_leaves"].tolist() == [4, 31, 255]
assert tabela_ex3["auc_ajuste"].is_monotonic_increasing, "a AUC de ajuste deve crescer com o número de folhas"
assert (tabela_ex3["auc_ajuste"] > tabela_ex3["auc_valid"]).all() and tabela_ex3["auc_valid"].between(0.7, 0.8).all()
print("Exercício 3 correto.")
tabela_ex3.assign(distancia=lambda t: t["auc_ajuste"] - t["auc_valid"]).round(4)

**Pergunta escrita.** Com base na sua tabela: qual dos três modelos você colocaria em produção? Qual está mais sobreajustado? O modelo com a melhor AUC de ajuste é o melhor modelo?

✏️ **Sua resposta:**

_Escreva aqui._

## 6. Trazendo o texto para a tabela

Até aqui o boosting não viu o conteúdo das avaliações, e a aula 05 mostrou que o sinal mais forte está lá. Uma forma simples de levar o texto a um modelo de árvores é criar **uma coluna por palavra**, com 1 se a palavra aparece na avaliação e 0 se não aparece. Ficamos com as 300 palavras mais frequentes.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

pre_palavras = ColumnTransformer(
    [
        ("num", "passthrough", NUMERICAS),
        ("cat", codificador(), CATEGORICAS),
        ("txt", CountVectorizer(max_features=300, binary=True), "texto"),          # vocabulário aprendido só no ajuste
    ],
    sparse_threshold=0,                                                          # devolve uma matriz densa comum
)
A = pre_palavras.fit_transform(X_ajuste).astype(np.float32)
V = pre_palavras.transform(X_valid).astype(np.float32)
nomes = nomes_das_colunas(pre_palavras)

print(A.shape, "· exemplos de colunas:", nomes[:3], "...", nomes[60:66])

In [ ]:
inicio = time.perf_counter()
lgbm = treinar_lgbm(A, V)
resultados["boosting (LightGBM) · tabular + 300 palavras"] = auc(lgbm, V, y_valid)
print(f"{lgbm.best_iteration_} árvores · AUC de validação {resultados['boosting (LightGBM) · tabular + 300 palavras']:.4f} · {time.perf_counter() - inicio:.1f} s")

In [ ]:
comparacao = pd.Series(resultados).sort_values()

fig, ax = plt.subplots(figsize=(8, 2.8))
cores = [AZUL if "boosting" in nome else CINZA for nome in comparacao.index]
ax.hlines(comparacao.index, 0.5, comparacao.values, color=cores, lw=1, alpha=0.5)
ax.scatter(comparacao.values, comparacao.index, color=cores, s=50, zorder=3)
for nome, valor in comparacao.items():
    ax.text(valor + 0.008, nome, f"{valor:.3f}", va="center", fontsize=9)
ax.set_xlim(0.5, 1.02); ax.set_xlabel("ROC AUC na validação (0,5 = sorteio)")
ax.set_title("O que entra no modelo pesa mais do que o algoritmo")
plt.tight_layout()

**Uma comparação honesta.**

- Nas **colunas tabulares**, o boosting ganha com folga da regressão logística.
- Com o **texto**, a regressão logística sobre o vocabulário completo (TF-IDF) continua na frente do boosting com 300 palavras, e é mais rápida e mais simples.
- A maior diferença do gráfico não está entre algoritmos: está entre **ter e não ter o texto**.

Isso é típico. Em dados esparsos e de dimensão muito alta, como contagens de palavras, cada coluna traz um pouco de informação e um modelo linear as soma com eficiência. Árvores brilham quando há poucas dezenas ou centenas de colunas densas, de tipos diferentes, com interações entre elas, o caso da maioria das tabelas de negócio.

Seguimos com o boosting de 300 palavras porque ele é um bom laboratório para os dois temas restantes: busca de hiperparâmetros e interpretação.

## 7. Optuna: busca de hiperparâmetros

Na aula 05 usamos `GridSearchCV`, que testa todas as combinações de uma grade. Com seis hiperparâmetros e cinco valores cada, seriam mais de 15 mil combinações. O **Optuna** faz uma busca adaptativa: usa o resultado das tentativas anteriores para decidir onde tentar em seguida.

| Conceito | O que é |
|---|---|
| **Estudo** (`study`) | uma busca completa, com direção (maximizar ou minimizar) |
| **Tentativa** (`trial`) | uma combinação de hiperparâmetros avaliada |
| **Função objetivo** | recebe uma tentativa, treina um modelo e devolve a nota |
| `trial.suggest_*` | pede um valor ao Optuna dentro de um intervalo |
| **Amostrador** | a estratégia de escolha; o padrão (TPE) concentra tentativas nas regiões promissoras |
| **Poda** | interrompe cedo as tentativas que já estão claramente piores |

Use `log=True` para hiperparâmetros cujo efeito é multiplicativo (taxas, penalidades, contagens que variam em ordens de grandeza): assim o Optuna explora 0,01, 0,1 e 1 com a mesma atenção.

⏳ A célula abaixo treina 20 modelos e leva de um a três minutos.

In [ ]:
N_TENTATIVAS = 20


def podar_se_ruim(tentativa):
    """Função chamada pelo LightGBM a cada árvore: informa o Optuna e interrompe as tentativas sem futuro."""
    def _a_cada_arvore(ambiente):
        auc_atual = ambiente.evaluation_result_list[0][2]
        tentativa.report(auc_atual, ambiente.iteration)
        if ambiente.iteration % 20 == 0 and tentativa.should_prune():
            raise optuna.TrialPruned()
    return _a_cada_arvore


def objetivo(tentativa):
    parametros = {
        "num_leaves": tentativa.suggest_int("num_leaves", 8, 128, log=True),
        "min_child_samples": tentativa.suggest_int("min_child_samples", 5, 200, log=True),
        "colsample_bytree": tentativa.suggest_float("colsample_bytree", 0.3, 1.0),
        "reg_lambda": tentativa.suggest_float("reg_lambda", 0.01, 30, log=True),
    }
    modelo = lgb.LGBMClassifier(n_estimators=2000, learning_rate=0.1, metric="auc", random_state=SEMENTE, verbose=-1, **parametros)
    modelo.fit(
        A, y_ajuste, **avaliar_em((V, y_valid)),                          # metric="auc": a parada antecipada e a poda usam a AUC
        callbacks=[lgb.early_stopping(50, verbose=False), podar_se_ruim(tentativa)],
    )
    tentativa.set_user_attr("arvores", modelo.best_iteration_)          # guardamos o número de árvores para usar depois
    return modelo.best_score_["valid_0"]["auc"]


inicio = time.perf_counter()
estudo = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=SEMENTE),                     # semente: a busca se repete
    pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=40),
)
estudo.optimize(objetivo, n_trials=N_TENTATIVAS)

print(f"{len(estudo.trials)} tentativas em {time.perf_counter() - inicio:.0f} s")
print(f"Melhor AUC de validação: {estudo.best_value:.4f}   (modelo com hiperparâmetros padrão: {resultados['boosting (LightGBM) · tabular + 300 palavras']:.4f})")
print("Melhores hiperparâmetros:", {k: round(v, 3) if isinstance(v, float) else v for k, v in estudo.best_params.items()})

In [ ]:
historico = estudo.trials_dataframe()
completas = historico[historico["state"] == "COMPLETE"]
podadas = historico[historico["state"] == "PRUNED"]

fig, eixos = plt.subplots(1, 2, figsize=(11.5, 3.4), gridspec_kw={"width_ratios": [1.5, 1]})
ax = eixos[0]
ax.scatter(completas["number"], completas["value"], color=AZUL, s=28, label="tentativa concluída", zorder=3)
ax.plot(completas["number"], completas["value"].cummax(), color=AZUL, lw=1, label="melhor até o momento")
for numero in podadas["number"]:
    ax.axvline(numero, color=CINZA, lw=0.8, ls=":")
ax.axhline(resultados["boosting (LightGBM) · tabular + 300 palavras"], color=LARANJA, lw=1, ls="--", label="hiperparâmetros padrão")
ax.set_xlabel("tentativa (linhas pontilhadas: tentativas podadas)"); ax.set_ylabel("AUC de validação")
ax.set_title("A busca melhora rápido no início e depois refina"); ax.legend(frameon=False, fontsize=8)

importancias = pd.Series(optuna.importance.get_param_importances(estudo)).sort_values()
eixos[1].barh(importancias.index, importancias.values, color=AZUL, height=0.6)
eixos[1].set_xlabel("participação na variação da nota"); eixos[1].set_title("Quais hiperparâmetros mais pesaram")
plt.tight_layout()
print(f"Tentativas podadas: {len(podadas)} de {len(historico)}")

### O ganho é real? Compensa?

A busca melhorou a AUC na terceira casa decimal. Antes de comemorar, duas perguntas.

**1. O ganho é maior que o ruído?** Uma forma barata de estimar o ruído é treinar o mesmo modelo com sementes diferentes (com sorteio de linhas e colunas ligado) e ver quanto a nota oscila.

In [ ]:
notas = []
for semente in range(3):
    m = lgb.LGBMClassifier(n_estimators=lgbm.best_iteration_, learning_rate=0.1, subsample=0.8, subsample_freq=1,
                           colsample_bytree=0.8, random_state=semente, verbose=-1).fit(A, y_ajuste)
    notas.append(auc(m, V, y_valid))

print("AUC com três sementes:", np.round(notas, 4), f"· amplitude {max(notas) - min(notas):.4f}")
print(f"Ganho da busca sobre o padrão: {estudo.best_value - resultados['boosting (LightGBM) · tabular + 300 palavras']:+.4f}")

O ganho da busca é algumas vezes maior que a oscilação entre sementes: é real, e é **pequeno**.

**2. Compensa?** Depende do uso. Em um sistema que decide milhões de casos por dia, dois milésimos de AUC podem valer muito. Em uma análise pontual, valem menos que o tempo gasto. Compare as ordens de grandeza desta aula:

| Decisão | Efeito na AUC |
|---|---|
| Incluir o texto | + 0,19 |
| Trocar regressão logística por boosting (colunas tabulares) | + 0,04 |
| Ajustar hiperparâmetros com 20 tentativas | + 0,002 |

> 📌 **Ordem de prioridades em um projeto:** primeiro os dados e as colunas, depois a validação correta, depois o tipo de modelo, e só então os hiperparâmetros.

> ⚠️ **Busca também sobreajusta.** Depois de 20 (ou 2000) tentativas avaliadas na mesma validação, a melhor nota é otimista: parte dela é sorte daquela divisão. A nota honesta do modelo escolhido continua sendo a do teste, que abriremos uma única vez.

### ✏️ Exercício 4 (★★★): sua própria busca

Escreva `objetivo_ex4`, uma função objetivo para o `HistGradientBoostingClassifier` na matriz **tabular** (`A_tab`, `V_tab`), e rode um estudo `estudo_ex4` com **15 tentativas**, maximizando a AUC de validação.

- Sugira `learning_rate` entre 0,02 e 0,3 (escala logarítmica), `max_leaf_nodes` entre 8 e 64, `min_samples_leaf` entre 10 e 200 e `l2_regularization` entre 0,001 e 10 (escala logarítmica).
- Use `max_iter=300`, `early_stopping=True` e `random_state=SEMENTE` no modelo.
- Crie o estudo com `sampler=optuna.samplers.TPESampler(seed=SEMENTE)`.

In [ ]:
# ✏️ Exercício 4
# SEU CÓDIGO AQUI
def objetivo_ex4(tentativa):
    ...


estudo_ex4 = ...

In [ ]:
# ✅ Verificação do exercício 4 (não altere)
assert len(estudo_ex4.trials) >= 15, "rode pelo menos 15 tentativas"
assert set(estudo_ex4.best_params) == {"learning_rate", "max_leaf_nodes", "min_samples_leaf", "l2_regularization"}, "use exatamente os quatro hiperparâmetros pedidos"
assert estudo_ex4.direction.name == "MAXIMIZE" and 0.75 < estudo_ex4.best_value < 0.80
_taxas = [t.params["learning_rate"] for t in estudo_ex4.trials]
assert 0.02 <= min(_taxas) and max(_taxas) <= 0.3
print(f"Exercício 4 correto: melhor AUC {estudo_ex4.best_value:.4f}  (HGB no padrão: {resultados['boosting (HGB) · tabular']:.4f})")

## 8. O que o modelo aprendeu?

Um modelo com centenas de árvores não cabe em uma tabela de pesos. Há três formas usuais de perguntar a ele quais colunas importam, e elas **não respondem à mesma pergunta**.

### 8.1 Importâncias internas: ganho e número de cortes

O LightGBM registra, para cada coluna, quantas vezes ela foi usada em um corte (`split`) e quanto esses cortes reduziram a perda (`gain`).

In [ ]:
por_ganho = pd.Series(lgbm.booster_.feature_importance("gain"), index=nomes).nlargest(8).sort_values()
por_cortes = pd.Series(lgbm.booster_.feature_importance("split"), index=nomes).nlargest(8).sort_values()

fig, eixos = plt.subplots(1, 2, figsize=(11.5, 3.2))
eixos[0].barh(por_ganho.index, por_ganho.values, color=AZUL, height=0.65)
eixos[0].set_title("Por ganho: quanto reduziu a perda"); eixos[0].set_xlabel("redução total da perda")
eixos[1].barh(por_cortes.index, por_cortes.values, color=CINZA, height=0.65)
eixos[1].set_title("Por número de cortes: quantas vezes foi usada"); eixos[1].set_xlabel("número de cortes")
plt.tight_layout()

As duas listas quase não têm colunas em comum. A contagem de cortes favorece colunas **numéricas com muitos valores distintos** (idade, tamanho do texto, proporção de maiúsculas), porque elas oferecem muitos pontos de corte e são usadas para pequenos ajustes finos. Uma coluna binária, como a presença de uma palavra, só pode ser cortada de um jeito, mesmo quando é decisiva.

> ⚠️ O `feature_importances_` do LightGBM usa, por padrão, o **número de cortes**. Em um relatório, essa lista diria que a idade é a coluna mais importante do modelo, e isso é falso.

### 8.2 Importância por permutação

A pergunta aqui é direta: **quanto o desempenho cai se eu embaralhar esta coluna?** Embaralhar destrói a relação da coluna com o alvo sem alterar sua distribuição. A medida usa dados de validação, vale para qualquer modelo e pode ser aplicada ao pipeline inteiro, o que dá a importância das colunas **originais**: o texto conta como uma coluna só.

In [ ]:
from sklearn.inspection import permutation_importance

pipeline_lgbm = Pipeline([
    ("pre", pre_palavras),
    ("clf", lgb.LGBMClassifier(n_estimators=lgbm.best_iteration_, learning_rate=0.1, random_state=SEMENTE, verbose=-1)),
]).fit(X_ajuste, y_ajuste)

colunas_originais = NUMERICAS + CATEGORICAS + ["texto"]
permutacao = permutation_importance(
    pipeline_lgbm, X_valid[colunas_originais].iloc[:5000], y_valid.iloc[:5000],
    scoring="roc_auc", n_repeats=3, random_state=SEMENTE,
)
pd.DataFrame({"queda na AUC": permutacao.importances_mean, "desvio": permutacao.importances_std}, index=colunas_originais).sort_values("queda na AUC", ascending=False).round(4).head(6)

Embaralhar o texto derruba a AUC em mais de 0,4, quase até o nível do sorteio. Embaralhar qualquer outra coluna muda a terceira casa decimal. (As três colunas derivadas do texto, como `palavras`, foram calculadas antes e não são afetadas pelo embaralhamento de `texto`.)

> ⚠️ **Limitação:** quando duas colunas carregam a mesma informação, embaralhar uma delas causa pouca queda, porque o modelo se apoia na outra. A importância por permutação pode subestimar colunas correlacionadas.

### 8.3 SHAP: a contribuição de cada coluna em cada previsão

As duas medidas anteriores resumem o modelo em um número por coluna. O **SHAP** responde a uma pergunta mais fina: para **esta** avaliação, quanto cada coluna empurrou a previsão para cima ou para baixo?

A ideia vem da teoria dos jogos (valores de Shapley): distribuir a previsão entre as colunas de forma justa, considerando a contribuição de cada uma em todas as combinações possíveis com as demais. Para modelos de árvores existe um algoritmo exato e rápido, o `TreeExplainer`.

A propriedade central é a **aditividade**:

> valor de base + soma dos valores SHAP da linha = previsão do modelo para a linha

No LightGBM de classificação, tudo isso é medido em **log-chances**. O valor de base é a previsão média do modelo.

In [ ]:
rng_amostra = np.random.default_rng(SEMENTE)
indices = rng_amostra.choice(len(V), size=2000, replace=False)          # 2000 avaliações da validação
V_amostra = V[indices]
textos_amostra = X_valid["texto"].iloc[indices].to_numpy()

explicador = shap.TreeExplainer(lgbm)
explicacao = explicador(V_amostra)
explicacao.feature_names = nomes

print("valores SHAP:", explicacao.values.shape, "(avaliações × colunas)")

# Conferindo a aditividade
probabilidades = lgbm.predict_proba(V_amostra)[:, 1]
log_chances = np.log(probabilidades / (1 - probabilidades))
reconstruido = explicacao.base_values + explicacao.values.sum(axis=1)
print(f"maior diferença entre a previsão e (base + soma dos SHAP): {np.abs(log_chances - reconstruido).max():.1e}")

#### Visão global

O gráfico abaixo tem um ponto por avaliação e por coluna. A posição horizontal é o valor SHAP (à direita: empurra para "não recomenda"); a cor é o valor da coluna naquela avaliação (vermelho: alto; azul: baixo).

In [ ]:
shap.plots.beeswarm(explicacao, max_display=12, show=False)
plt.gcf().set_size_inches(8.5, 4.6)
plt.xlabel("valor SHAP (efeito na previsão, em log-chances)")
plt.title("Contribuição de cada coluna, avaliação por avaliação")
plt.tight_layout()

**Como ler.**

- As colunas estão ordenadas pela média do valor absoluto do SHAP: as de cima são as que mais movem as previsões.
- Em uma coluna de palavra, os pontos vermelhos são as avaliações que **contêm** a palavra e os azuis, as que não contêm.
- `palavra: não` é a mais influente: quando presente, empurra a previsão para "não recomenda"; quando ausente, empurra um pouco para o lado oposto.
- Palavras de elogio ("bom", "excelente", "ótimo") têm o padrão inverso.

O SHAP mostra **direção e tamanho** do efeito, coisa que as importâncias das seções anteriores não mostram.

#### Dependência: o efeito de uma coluna ao longo dos seus valores

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.6))
shap.plots.scatter(explicacao[:, "palavras"], color=explicacao[:, "palavra: não"], ax=ax, show=False, alpha=0.5, dot_size=8)
ax.set_xlim(0, 150); ax.set_xlabel("número de palavras da avaliação"); ax.set_ylabel("valor SHAP de `palavras`")
ax.set_title("Avaliações longas empurram para “não recomenda”")
plt.tight_layout()

O efeito do tamanho do texto **não é uma reta**, e nem sequer cresce sempre:

- avaliações de cerca de dez palavras puxam a previsão para "recomenda";
- daí em diante, quanto mais longo o texto, mais a previsão é empurrada para "não recomenda" (quem está insatisfeito escreve mais), até estabilizar;
- no extremo esquerdo, um grupo de textos de pouquíssimas palavras tem efeito na direção de "não recomenda".

É o tipo de relação que as árvores capturam sem que ninguém precise descrevê-la, e que explica parte da vantagem sobre o modelo linear nas colunas tabulares. A cor mostra a presença da palavra "não"; os pontos vermelhos e azuis se misturam ao longo da curva, sinal de que as duas colunas pouco interagem.

#### Uma previsão explicada

Vamos pegar a avaliação da amostra com a maior probabilidade prevista de "não recomenda".

In [ ]:
mais_provavel = int(np.argmax(probabilidades))
print(f"Probabilidade prevista: {probabilidades[mais_provavel]:.3f}")
print("Texto:", textos_amostra[mais_provavel][:400])

shap.plots.waterfall(explicacao[mais_provavel], max_display=9, show=False)
plt.gcf().set_size_inches(8.5, 4.2)
plt.tight_layout()
plt.show()

O gráfico parte, embaixo, do valor de base `E[f(X)]` (a previsão média, em log-chances) e sobe até `f(x)`, a previsão desta avaliação. Cada barra é a contribuição de uma coluna; à esquerda de cada nome aparece o valor da coluna (1 = a palavra está no texto).

Dois detalhes para notar. As **ausências** também contribuem: a falta de "bom" e de "excelente" empurra a previsão para cima. E o valor de base é bem negativo, porque é uma média em log-chances: a maioria das avaliações recebe probabilidades pequenas, que correspondem a log-chances muito negativas.

### O que o SHAP não diz

| Cuidado | Por quê |
|---|---|
| **Explica o modelo, e não o mundo** | se o modelo aprendeu um atalho ou um vazamento, o SHAP o mostrará como "importante" |
| **Não é causalidade** | a palavra "não" não causa insatisfação; ela a acompanha |
| **Colunas correlacionadas dividem o crédito** | "não" e "nao" repartem um efeito que é um só |
| **A unidade é log-chances** | um valor SHAP de 1 não significa "mais 1 ponto de probabilidade" |
| **O valor depende da amostra de referência** | mudar os dados de base muda o valor de base e os SHAP |

### ✏️ Exercício 5 (★★): explicar uma previsão

Considere a avaliação da amostra com a **menor** probabilidade prevista de "não recomenda".

- guarde a posição dela em `i_ex5`;
- guarde em `top3_ex5` uma lista com os nomes das três colunas de maior valor SHAP **absoluto** nessa avaliação, da maior para a menor.

In [ ]:
# ✏️ Exercício 5
# SEU CÓDIGO AQUI
i_ex5 = ...
top3_ex5 = ...

In [ ]:
# ✅ Verificação do exercício 5 (não altere)
assert probabilidades[i_ex5] == probabilidades.min(), "i_ex5 deve apontar para a menor probabilidade prevista"
assert len(top3_ex5) == 3 and all(nome in nomes for nome in top3_ex5), "top3_ex5 deve ter três nomes de colunas"
_abs = np.abs(explicacao.values[i_ex5])
_dos_tres = [_abs[nomes.index(nome)] for nome in top3_ex5]
assert min(_dos_tres) >= np.sort(_abs)[-3] - 1e-12 and _dos_tres == sorted(_dos_tres, reverse=True), "não são as três maiores contribuições, ou estão fora de ordem"
print("Exercício 5 correto:", top3_ex5)

### ✏️ Exercício 6 (★★): leitura crítica

Responda por escrito, com base nos gráficos da seção 8.

**a)** No gráfico global, por que os pontos **azuis** de `palavra: não` ficam um pouco à esquerda do zero, e não exatamente em zero?

**b)** Um gerente lê o gráfico e propõe: "a palavra *não* é o que mais gera não recomendação; vamos orientar o atendimento a evitar que os clientes a usem". O que há de errado no raciocínio?

**c)** As três medidas de importância (número de cortes, ganho, permutação) e o SHAP apontaram colunas diferentes como as mais importantes. Qual você levaria para um relatório, e por quê?

✏️ **Sua resposta:**

_Escreva aqui._

## 9. Abrindo o cofre

As escolhas estão feitas: matriz com 300 palavras e os hiperparâmetros encontrados pelo Optuna. Treinamos o pipeline final em **todo o treino** (ajuste e validação juntos) e medimos no teste, uma única vez.

Sem validação separada não há parada antecipada. Usamos o número de árvores da melhor tentativa, com um acréscimo de 10% porque agora há mais dados.

In [ ]:
melhor = estudo.best_trial
arvores_finais = int(melhor.user_attrs["arvores"] * 1.1)

modelo_final = Pipeline([
    ("pre", pre_palavras),
    ("clf", lgb.LGBMClassifier(n_estimators=arvores_finais, learning_rate=0.1, random_state=SEMENTE, verbose=-1, **melhor.params)),
]).fit(X_treino, y_treino)

print(f"Boosting ajustado ({arvores_finais} árvores) · AUC no TESTE: {auc(modelo_final, X_teste, y_teste):.4f}")
print(f"Estimativa na validação durante a busca:            {estudo.best_value:.4f}")

In [ ]:
import joblib

joblib.dump(modelo_final, "modelo_boosting.joblib")
print(f"modelo_boosting.joblib: {Path('modelo_boosting.joblib').stat().st_size / 1e6:.1f} MB")

A nota no teste fica próxima da estimada na validação, sinal de que o processo foi honesto. O pipeline com regressão logística e TF-IDF da aula 05 obteve 0,965 no mesmo teste, e continua sendo o melhor modelo para este problema: mais simples, mais rápido e um pouco mais preciso.

Saber chegar a essa conclusão, com números, é parte do trabalho.

## 10. Para levar

### Quando usar boosting

| Situação | Recomendação |
|---|---|
| Tabela com colunas numéricas e categóricas misturadas | **boosting** como primeira tentativa séria |
| Texto, com milhares de colunas esparsas | modelo linear sobre TF-IDF; depois, modelos de linguagem (aulas 09 e 10) |
| Imagens, áudio, sequências | redes neurais (aulas 07 e 08) |
| Poucas centenas de linhas | modelos simples e bem regularizados; o boosting sobreajusta com facilidade |
| Exigência de modelo explicável por regulação | modelo linear ou árvore pequena; ou boosting com SHAP, se aceito |
| Extrapolar além dos valores vistos (tendências) | árvores não extrapolam; prefira modelos lineares ou trate a tendência antes |

### Erros comuns

| Erro | Como evitar |
|---|---|
| Fixar `n_estimators` no chute | número alto com parada antecipada |
| Usar o teste como conjunto de parada antecipada | a parada usa a validação; o teste fica fora |
| Escolher o modelo pela nota de ajuste | comparar sempre na validação |
| Reportar a melhor nota da busca como desempenho final | a nota final é a do teste, medida uma vez |
| Aplicar *one-hot* em categóricas de alta cardinalidade para árvores | códigos inteiros ou suporte nativo a categóricas |
| Usar `feature_importances_` (número de cortes) em relatório | ganho, permutação ou SHAP |
| Ler SHAP como causa | SHAP explica o modelo; causalidade exige outro tipo de estudo |
| Horas de busca de hiperparâmetros antes de olhar as colunas | dados e colunas primeiro |

### Resumo

- Boosting soma árvores rasas; cada uma é treinada no **resíduo** das anteriores, com um passo controlado pela **taxa de aprendizado**.
- Mais árvores reduzem sempre o erro de ajuste; o de validação tem um mínimo. A **parada antecipada** encontra esse ponto.
- `num_leaves` e `min_child_samples` controlam a complexidade das árvores; sorteio de colunas e penalidades regularizam.
- O **Optuna** busca hiperparâmetros de forma adaptativa e poda tentativas ruins. O ganho costuma ser pequeno diante do efeito das colunas.
- Importância por número de cortes engana; **permutação** mede a queda de desempenho; **SHAP** dá direção e tamanho do efeito em cada previsão.
- Em tabelas, boosting é a escolha padrão. Em texto esparso, um modelo linear compete de igual para igual.

### O que vem a seguir

O notebook 07 começa o PyTorch: tensores, diferenciação automática e a construção de uma rede neural desde o início.

### Para casa

A lista de exercícios está em `06_casa.ipynb`.